# NiyamTrace-X — External Multi-Benchmark Testing V2 (Fixed)

This notebook is **external-only** and incorporates the exact failures observed in the uploaded result package.

### Fixes applied

- **BFCL V4:** uses EvalScope's BFCL-v4 integration against the same OpenAI-compatible/OpenRouter endpoints; no `python -m bfcl_eval` assumption.
- **AgentDojo:** uses the installed `openai-compatible` provider + `--model-id`, which the observed CLI actually supports.
- **AgentDyn:** uses a unique output directory and `--force-rerun`, preventing cached/repository results from being mistaken for a fresh run.
- **τ³:** installs `websockets` and `soundfile` explicitly into the repository's own `.venv`, then verifies the imports before evaluation.
- **MCP-SafetyBench:** isolated and disabled by default because it can touch real external systems.
- **All results:** benchmark-native metrics only; installation success is never counted as evidence.

Start with `QUICK`.

In [ ]:
# CELL 1 — SETUP
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os,sys,json,re,time,random,hashlib,zipfile,shutil,subprocess,glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED)

BASE=Path("/content/NTX_EXTERNAL_V2") if Path("/content").exists() else Path.cwd()/"NTX_EXTERNAL_V2"
WORK=BASE/"work"; RESULTS=BASE/"results"; RAW=RESULTS/"raw"
for p in [BASE,WORK,RESULTS,RAW]:p.mkdir(parents=True,exist_ok=True)

MODE=os.getenv("NTX_RUN_MODE","QUICK").upper()
assert MODE in {"QUICK","STANDARD","FULL"}
LIMIT={"QUICK":10,"STANDARD":100,"FULL":None}[MODE]
DOJO_USER_LIMIT={"QUICK":2,"STANDARD":10,"FULL":None}[MODE]
TAU_LIMIT={"QUICK":3,"STANDARD":20,"FULL":None}[MODE]

def run(cmd,cwd=None,env=None,timeout=None,check=False):
    p=subprocess.run(cmd,cwd=cwd,env=env,capture_output=True,text=True,errors="replace",timeout=timeout)
    if check and p.returncode!=0:raise RuntimeError((p.stderr or p.stdout)[-4000:])
    return p

def log(name,p):
    (RESULTS/name).write_text((p.stdout or "")+"\nSTDERR\n"+(p.stderr or ""))

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""):h.update(c)
    return h.hexdigest()

def ensure_uv():
    if not shutil.which("uv"):run([sys.executable,"-m","pip","install","-q","uv"],check=True)

def clone(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=run(["git","clone","--depth","1",url,str(dest)])
        if p.returncode:raise RuntimeError(p.stderr[-3000:])
    return run(["git","-C",str(dest),"rev-parse","HEAD"],check=True).stdout.strip()

print("Mode:",MODE,"Base:",BASE)

In [ ]:
# CELL 2 — ONE OPENROUTER KEY / THREE FAMILIES
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY","").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY=getpass("OpenRouter API key (hidden): ").strip()
if not OPENROUTER_API_KEY:raise RuntimeError("OPENROUTER_API_KEY required.")

MODELS=[
{"label":"qwen","family":"Qwen","model":"qwen/qwen3-coder:exacto"},
{"label":"gpt-oss","family":"GPT-OSS","model":"openai/gpt-oss-120b:exacto"},
{"label":"glm","family":"GLM","model":"z-ai/glm-4.6:exacto"},
]
API_URL="https://openrouter.ai/api/v1"
print([(m["label"],m["family"],m["model"]) for m in MODELS])

In [ ]:
# CELL 3 — PROBE ALL ENDPOINTS
import urllib.request
probes=[]
for m in MODELS:
    req=urllib.request.Request(API_URL+"/chat/completions",
        data=json.dumps({"model":m["model"],"messages":[{"role":"user","content":"Reply exactly OK"}],"temperature":0,"max_tokens":8}).encode(),
        headers={"Authorization":"Bearer "+OPENROUTER_API_KEY,"Content-Type":"application/json"})
    t=time.time()
    try:
        with urllib.request.urlopen(req,timeout=45) as r:o=json.loads(r.read().decode())
        probes.append({"model":m["label"],"family":m["family"],"status":"OK","latency_s":time.time()-t,
                       "reply":str(o.get("choices",[{}])[0].get("message",{}).get("content",""))[:80]})
    except Exception as e:
        probes.append({"model":m["label"],"family":m["family"],"status":"ERROR","error":repr(e)})
probe=pd.DataFrame(probes);probe.to_csv(RESULTS/"00_endpoint_probe.csv",index=False);display(probe)
WORKING=[m for m in MODELS if len(probe[(probe.model==m["label"])&(probe.status=="OK")])]
if not WORKING:raise RuntimeError("No endpoints reachable.")
if MODE=="FULL" and len({m["family"] for m in WORKING})<3:raise RuntimeError("FULL run requires all three model families.")

# A. BFCL V4 through EvalScope

EvalScope currently exposes `bfcl_v4` directly for OpenAI-compatible endpoints.
This avoids the BFCL CLI/module mismatch that occurred in the uploaded run.

QUICK: 10 cases/model  
STANDARD: 100 cases/model  
FULL: no limit

In [ ]:
# CELL 4 — INSTALL + RUN BFCL-V4
BFCL_ENV=WORK/"bfcl_evalscope"
ensure_uv()
run(["uv","python","install","3.12"])
if not BFCL_ENV.exists():run(["uv","venv",str(BFCL_ENV),"--python","3.12"],check=True)
BP=str(BFCL_ENV/"bin"/"python")
ip=run([BP,"-m","pip","install","-q","evalscope","bfcl-eval==2025.10.27.1","soundfile","websockets"])
log("10_bfcl_install.log",ip)
if ip.returncode:raise RuntimeError("BFCL/EvalScope install failed.")

bfcl_rows=[]
BFCL_OUT=RAW/"bfcl"
BFCL_OUT.mkdir(parents=True,exist_ok=True)

for m in WORKING:
    work=BFCL_OUT/m["label"];work.mkdir(parents=True,exist_ok=True)
    script=[
        "from evalscope import run_task",
        "from evalscope.config import TaskConfig",
        "cfg=TaskConfig(",
        f" model={m['model']!r},",
        f" api_url={API_URL!r},",
        f" api_key={OPENROUTER_API_KEY!r},",
        " eval_type='openai_api',",
        " datasets=['bfcl_v4'],",
        f" work_dir={str(work)!r},",
        (" limit=None," if LIMIT is None else f" limit={LIMIT},"),
        " dataset_args={'bfcl_v4': {'extra_params': {'is_fc_model': True}}},",
        ")",
        "run_task(task_cfg=cfg)",
    ]
    py=work/"run_bfcl.py";py.write_text("\n".join(script))
    p=run([BP,str(py)],cwd=work,timeout=None)
    log(f"10_bfcl_{m['label']}.log",p)
    bfcl_rows.append({"benchmark":"BFCL-v4","model":m["label"],"family":m["family"],
                      "status":"OK" if p.returncode==0 else "ERROR","returncode":p.returncode})

pd.DataFrame(bfcl_rows).to_csv(RESULTS/"10_bfcl_run_status.csv",index=False)
display(pd.DataFrame(bfcl_rows))

In [ ]:
# CELL 5 — PARSE BFCL NUMERIC REPORTS
bfcl_metrics=[]
for m in WORKING:
    root=RAW/"bfcl"/m["label"]
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".json",".jsonl",".csv"}:continue
        try:
            if p.suffix==".csv":
                d=pd.read_csv(p)
                for col in d.columns:
                    if any(k in col.lower() for k in ["accuracy","score"]):
                        vals=pd.to_numeric(d[col],errors="coerce").dropna()
                        for v in vals:
                            bfcl_metrics.append({"benchmark":"BFCL-v4","model":m["label"],"family":m["family"],
                                                 "metric":col,"score":float(v),"source":str(p.relative_to(root))})
            else:
                txt=p.read_text(errors="ignore")
                objs=[]
                try:objs=[json.loads(txt)]
                except Exception:
                    for line in txt.splitlines():
                        try:objs.append(json.loads(line))
                        except Exception:pass
                def walk(o,path=""):
                    if isinstance(o,dict):
                        for k,v in o.items():
                            q=f"{path}.{k}" if path else k
                            if isinstance(v,(dict,list)):walk(v,q)
                            elif isinstance(v,(int,float)) and any(x in k.lower() for x in ["accuracy","score"]):
                                bfcl_metrics.append({"benchmark":"BFCL-v4","model":m["label"],"family":m["family"],
                                                     "metric":q,"score":float(v),"source":str(p.relative_to(root))})
                    elif isinstance(o,list):
                        for i,v in enumerate(o):walk(v,f"{path}[{i}]")
                for o in objs:walk(o)
        except Exception:pass

bfcl_metrics=pd.DataFrame(bfcl_metrics).drop_duplicates() if bfcl_metrics else pd.DataFrame()
bfcl_metrics.to_csv(RESULTS/"10_bfcl_metrics.csv",index=False)
display(bfcl_metrics.head(50))

# B. AgentDojo — OpenAI-compatible provider

The uploaded run proved that the installed AgentDojo accepts `openai-compatible`, but not `QWEN3_235B`.

This stage therefore runs all three OpenRouter models through:

- `--model openai-compatible`
- `--model-id <exact OpenRouter model>`
- `OPENAI_COMPATIBLE_BASE_URL`
- `OPENAI_COMPATIBLE_API_KEY`
- unique `--logdir`
- `--force-rerun`

In [ ]:
# CELL 6 — AGENTDOJO
DOJO=WORK/"agentdojo";dojo_rows=[]
commit=clone("https://github.com/ethz-spylab/agentdojo.git",DOJO);ensure_uv()
log("11_dojo_sync.log",run(["uv","sync"],cwd=DOJO))
helpout=run(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO)
log("11_dojo_help.log",helpout)
if "openai-compatible" not in ((helpout.stdout or "")+(helpout.stderr or "")):
    raise RuntimeError("Installed AgentDojo does not expose openai-compatible.")

suites=["banking"] if MODE=="QUICK" else (["banking","workspace"] if MODE=="STANDARD" else ["banking","workspace","travel","slack"])
for m in WORKING:
    for suite in suites:
        out=RAW/"agentdojo"/m["label"]/suite/f"run_{int(time.time()*1000)}";out.mkdir(parents=True,exist_ok=True)
        env=os.environ.copy()
        env["OPENAI_COMPATIBLE_BASE_URL"]=API_URL
        env["OPENAI_COMPATIBLE_API_KEY"]=OPENROUTER_API_KEY
        cmd=["uv","run","python","-m","agentdojo.scripts.benchmark",
             "--model","openai-compatible","--model-id",m["model"],
             "-s",suite,"--attack","important_instructions",
             "--logdir",str(out),"--force-rerun","--max-workers","1"]
        if DOJO_USER_LIMIT is not None:
            for i in range(DOJO_USER_LIMIT):cmd+=["-ut",f"user_task_{i}"]
        p=run(cmd,cwd=DOJO,env=env,timeout=None)
        log(f"11_dojo_{m['label']}_{suite}.log",p)
        txt=(p.stdout or "")+"\n"+(p.stderr or "")
        util=re.findall(r"Average utility:\s*([0-9.]+)%",txt)
        sec=re.findall(r"Average security:\s*([0-9.]+)%",txt)
        inj=re.findall(r"Passed injection tasks as user tasks:\s*(\d+)/(\d+)",txt)
        dojo_rows.append({"benchmark":"AgentDojo","model":m["label"],"family":m["family"],"suite":suite,
                          "utility":float(util[-1])/100 if util else np.nan,
                          "security":float(sec[-1])/100 if sec else np.nan,
                          "inj_passed":int(inj[-1][0]) if inj else np.nan,
                          "inj_total":int(inj[-1][1]) if inj else np.nan,
                          "status":"OK" if p.returncode==0 and (util or sec) else "ERROR","returncode":p.returncode,
                          "commit":commit})
dojo=pd.DataFrame(dojo_rows);dojo.to_csv(RESULTS/"11_agentdojo_metrics.csv",index=False);display(dojo)

# C. AgentDyn — forced fresh run

The uploaded AgentDyn log reported that tasks were **already run**, so its 0% utility / 2.78% security cannot be treated as a fresh result.

This stage always uses a unique `--logdir` and `--force-rerun`.
It dynamically uses `openai-compatible` if the installed AgentDyn CLI exposes it; otherwise it records the model as unsupported instead of reusing cached data.

In [ ]:
# CELL 7 — AGENTDYN
AD=WORK/"AgentDyn";ad_rows=[]
commit=clone("https://github.com/SaFo-Lab/AgentDyn.git",AD)
log("12_agentdyn_install.log",run([sys.executable,"-m","pip","install","-q","-e",str(AD)]))
helpout=run([sys.executable,"-m","agentdojo.scripts.benchmark","--help"],cwd=AD)
log("12_agentdyn_help.log",helpout)
helptext=(helpout.stdout or "")+(helpout.stderr or "")
can_compat="openai-compatible" in helptext

suites=["shopping"] if MODE=="QUICK" else (["shopping","github"] if MODE=="STANDARD" else ["shopping","github","dailylife"])
for m in WORKING:
    if not can_compat:
        ad_rows.append({"benchmark":"AgentDyn","model":m["label"],"family":m["family"],"suite":"ALL",
                        "status":"UNSUPPORTED_OPENAI_COMPATIBLE_NOT_EXPOSED","commit":commit})
        continue
    for suite in suites:
        out=RAW/"agentdyn"/m["label"]/suite/f"run_{int(time.time()*1000)}";out.mkdir(parents=True,exist_ok=True)
        env=os.environ.copy()
        env["OPENAI_COMPATIBLE_BASE_URL"]=API_URL
        env["OPENAI_COMPATIBLE_API_KEY"]=OPENROUTER_API_KEY
        cmd=[sys.executable,"-m","agentdojo.scripts.benchmark",
             "--model","openai-compatible","--model-id",m["model"],
             "-s",suite,"--attack","important_instructions",
             "--logdir",str(out),"--force-rerun","--max-workers","1"]
        if DOJO_USER_LIMIT is not None:
            for i in range(DOJO_USER_LIMIT):cmd+=["-ut",f"user_task_{i}"]
        p=run(cmd,cwd=AD,env=env,timeout=None)
        log(f"12_agentdyn_{m['label']}_{suite}.log",p)
        txt=(p.stdout or "")+"\n"+(p.stderr or "")
        util=re.findall(r"Average utility:\s*([0-9.]+)%",txt)
        sec=re.findall(r"Average security:\s*([0-9.]+)%",txt)
        inj=re.findall(r"Passed injection tasks as user tasks:\s*(\d+)/(\d+)",txt)
        ad_rows.append({"benchmark":"AgentDyn","model":m["label"],"family":m["family"],"suite":suite,
                        "utility":float(util[-1])/100 if util else np.nan,
                        "security":float(sec[-1])/100 if sec else np.nan,
                        "inj_passed":int(inj[-1][0]) if inj else np.nan,
                        "inj_total":int(inj[-1][1]) if inj else np.nan,
                        "status":"OK" if p.returncode==0 and (util or sec) else "ERROR",
                        "returncode":p.returncode,"commit":commit})
agentdyn=pd.DataFrame(ad_rows);agentdyn.to_csv(RESULTS/"12_agentdyn_metrics.csv",index=False);display(agentdyn)

# D. τ³ / tau2-bench — venv dependency fix

The uploaded traceback is unambiguous: the benchmark's own `.venv` could not import `websockets`.

This stage installs and verifies the required packages **inside that exact `.venv`** before any task is started.

In [ ]:
# CELL 8 — TAU INSTALL / DEPENDENCY VERIFICATION / RUN
TAU=WORK/"tau2-bench";tau_rows=[]
commit=clone("https://github.com/sierra-research/tau2-bench.git",TAU);ensure_uv()
log("14_tau_sync.log",run(["uv","sync"],cwd=TAU))
TAUPY=TAU/".venv"/"bin"/"python"
if not TAUPY.exists():raise RuntimeError("tau2 .venv Python not found after uv sync.")

dep=run(["uv","pip","install","--python",str(TAUPY),"websockets","soundfile"],cwd=TAU)
log("14_tau_dependency_install.log",dep)
verify=run([str(TAUPY),"-c","import websockets,soundfile; print('TAU_DEPS_OK')"],cwd=TAU)
log("14_tau_dependency_verify.log",verify)
if verify.returncode!=0:raise RuntimeError("tau2 dependency verification failed.")

domains=["airline","retail","telecom"]
for m in WORKING:
    for domain in domains:
        tag=f"ntxv2_{m['label']}_{domain}_{int(time.time()*1000)}"
        env=os.environ.copy();env["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY
        cmd=["uv","run","tau2","run","--domain",domain,
             "--agent-llm","openrouter/"+m["model"],"--user-llm","openrouter/"+m["model"],
             "--num-trials","1","--task-split-name","base","--max-concurrency","1",
             "--seed",str(SEED),"--save-to",tag]
        if TAU_LIMIT is not None:cmd+=["--num-tasks",str(TAU_LIMIT)]
        p=run(cmd,cwd=TAU,env=env,timeout=None)
        log(f"14_tau_{m['label']}_{domain}.log",p)
        tau_rows.append({"benchmark":"tau3","model":m["label"],"family":m["family"],"domain":domain,
                         "status":"OK" if p.returncode==0 else "ERROR","returncode":p.returncode,
                         "tag":tag,"commit":commit})
tau_runs=pd.DataFrame(tau_rows);tau_runs.to_csv(RESULTS/"14_tau_run_status.csv",index=False);display(tau_runs)

In [ ]:
# CELL 9 — TAU PER-TRAJECTORY PARSER
def deep_numeric(o,keys):
    if isinstance(o,dict):
        for k in keys:
            if k in o and isinstance(o[k],(int,float,bool)):return float(o[k])
        for v in o.values():
            x=deep_numeric(v,keys)
            if x is not None:return x
    elif isinstance(o,list):
        for v in o:
            x=deep_numeric(v,keys)
            if x is not None:return x
    return None

tau_traj=[]
simroot=TAU/"data"/"simulations"
if not simroot.exists():simroot=TAU/"data"/"tau2"/"simulations"
for _,rr in tau_runs[tau_runs.status=="OK"].iterrows():
    for p in simroot.rglob(f"*{rr.tag}*.json") if simroot.exists() else []:
        try:o=json.loads(p.read_text())
        except Exception:continue
        items=o if isinstance(o,list) else next((o[k] for k in ["simulations","results","trajectories","data"] if isinstance(o,dict) and isinstance(o.get(k),list)),[o])
        for i,item in enumerate(items):
            if not isinstance(item,dict):continue
            reward=deep_numeric(item,["reward","score","success","task_success"])
            tau_traj.append({"benchmark":"tau3","model":rr.model,"family":rr.family,"domain":rr.domain,
                             "trajectory_index":i,"reward":reward,"source_file":str(p.relative_to(TAU))})
tau=pd.DataFrame(tau_traj);tau.to_csv(RESULTS/"14_tau_trajectory_metrics.csv",index=False)
tau_summary=(tau.groupby(["benchmark","model","family","domain"],dropna=False)
             .agg(n=("reward","count"),mean_reward=("reward","mean")).reset_index()) if len(tau) else pd.DataFrame()
tau_summary.to_csv(RESULTS/"14_tau_summary.csv",index=False);display(tau_summary)

# E. MCP-SafetyBench

Kept disabled by default and isolated from the Colab environment.

Enable only if you intentionally want the real-MCP tests.

In [ ]:
# CELL 10 — MCP SAFE / ISOLATED GATE
ENABLE_MCP=False
mcp_rows=[]
if ENABLE_MCP:
    MCP=WORK/"MCPSafety";commit=clone("https://github.com/xjzzzzzzzz/MCPSafety.git",MCP);ensure_uv()
    MENV=WORK/"mcp_py312";run(["uv","python","install","3.12"])
    if not MENV.exists():run(["uv","venv",str(MENV),"--python","3.12"],check=True)
    MPY=str(MENV/"bin"/"python")
    p=run([MPY,"-m","pip","install","-q","-r",str(MCP/"requirements.txt")]);log("13_mcp_install.log",p)
    mcp_rows.append({"benchmark":"MCP-SafetyBench","status":"READY_ISOLATED_NOT_AUTORUN","commit":commit})
else:
    mcp_rows=[{"benchmark":"MCP-SafetyBench","status":"DISABLED_BY_DEFAULT"}]
pd.DataFrame(mcp_rows).to_csv(RESULTS/"13_mcp_status.csv",index=False);display(pd.DataFrame(mcp_rows))

In [ ]:
# CELL 11 — UNIFIED BENCHMARK-NATIVE METRICS
rows=[]

if len(bfcl_metrics):
    for _,r in bfcl_metrics.iterrows():
        if pd.notna(r.score):
            rows.append({"benchmark":"BFCL-v4","model":r.model,"family":r.family,"slice":r.metric,
                         "metric":"accuracy_or_score","score":r.score,"paper_eligible":True})

if len(dojo):
    for _,r in dojo[dojo.status=="OK"].iterrows():
        if pd.notna(r.utility):rows.append({"benchmark":"AgentDojo","model":r.model,"family":r.family,"slice":r.suite,"metric":"utility","score":r.utility,"paper_eligible":True})
        if pd.notna(r.security):rows.append({"benchmark":"AgentDojo","model":r.model,"family":r.family,"slice":r.suite,"metric":"security","score":r.security,"paper_eligible":True})

if len(agentdyn):
    for _,r in agentdyn[agentdyn.status=="OK"].iterrows():
        if pd.notna(r.utility):rows.append({"benchmark":"AgentDyn","model":r.model,"family":r.family,"slice":r.suite,"metric":"utility","score":r.utility,"paper_eligible":True})
        if pd.notna(r.security):rows.append({"benchmark":"AgentDyn","model":r.model,"family":r.family,"slice":r.suite,"metric":"security","score":r.security,"paper_eligible":True})

if len(tau_summary):
    for _,r in tau_summary.iterrows():
        if pd.notna(r.mean_reward):rows.append({"benchmark":"tau3","model":r.model,"family":r.family,"slice":r.domain,"metric":"mean_reward","score":r.mean_reward,"paper_eligible":True})

unified=pd.DataFrame(rows)
unified.to_csv(RESULTS/"20_external_unified_evidence.csv",index=False)
display(unified)

In [ ]:
# CELL 12 — CLAIM CHECKLIST / FIGURES
checks=[]
for b in ["BFCL-v4","AgentDojo","AgentDyn","tau3"]:
    checks.append({"claim":f"{b} numeric external evidence","status":"SUPPORTED" if len(unified[unified.benchmark==b]) else "MISSING"})
benches=set(unified.benchmark) if len(unified) else set()
fams=set(unified.family)-{""} if len(unified) else set()
checks += [
{"claim":">=3 external benchmark families","status":"SUPPORTED" if len(benches)>=3 else "MISSING"},
{"claim":">=3 external model families","status":"SUPPORTED" if len(fams)>=3 else "MISSING"}]
claims=pd.DataFrame(checks);claims.to_csv(RESULTS/"22_external_claim_checklist.csv",index=False);display(claims)

if len(unified):
    for (b,metric),g in unified.groupby(["benchmark","metric"]):
        gg=g.groupby("model",as_index=False).score.mean().sort_values("score")
        fig,ax=plt.subplots(figsize=(8,max(3,0.5*len(gg)+1)))
        ax.barh(gg.model,gg.score);ax.set_title(f"{b}: {metric}");ax.set_xlabel(metric);fig.tight_layout()
        fig.savefig(RESULTS/f"figure_{re.sub('[^A-Za-z0-9]+','_',b)}_{re.sub('[^A-Za-z0-9]+','_',metric)}.png",dpi=220)
        plt.show()

In [ ]:
# CELL 13 — MANIFEST / RESULT ZIP DOWNLOAD
manifest={"experiment":"NTX-EXTERNAL-V2-FIXED","created_at":datetime.now().isoformat(),"mode":MODE,
          "models":WORKING,"claim_checklist":claims.to_dict("records")}
hashes={}
for p in RESULTS.rglob("*"):
    if p.is_file() and p.name not in {"FINAL_MANIFEST.json","SHA256SUMS.txt"}:hashes[str(p.relative_to(RESULTS))]=sha256_file(p)
manifest["artifact_sha256"]=hashes
(RESULTS/"FINAL_MANIFEST.json").write_text(json.dumps(manifest,indent=2,default=str))
(RESULTS/"SHA256SUMS.txt").write_text("\n".join(f"{h}  {k}" for k,h in sorted(hashes.items()))+"\n")
zipout=BASE/"NTX_EXTERNAL_MULTI_BENCHMARK_V2_RESULTS.zip"
if zipout.exists():zipout.unlink()
with zipfile.ZipFile(zipout,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.rglob("*"):
        if p.is_file():z.write(p,arcname=str(p.relative_to(RESULTS)))
print("ZIP:",zipout)
print("SHA256:",sha256_file(zipout))
display(claims)
try:
    from google.colab import files;files.download(str(zipout))
except Exception:pass